# DeBERTa-v3-base emotion classifier

Fine-tune Microsoft's DeBERTa-v3-base on the `text`, `emo_1` and `emo_2`
columns of `aggr_ds.csv`. Each word or sentence produces a nine-element
vector of independent sigmoid confidences, one per label, in this order:

**Anger, Anticipation, Disgust, Fear, Joy, Sadness, Surprise, Trust, Neutral**

Because the confidences are independent, a text can score high on several
labels at once, and the nine values do not have to add up to 1.

Open in Jupyter or Colab, run the setup cell, and run the remaining cells in
order. In Colab, select a GPU runtime and upload the CSV to the working
directory. In local Jupyter, put the CSV beside the notebook or set
`DATA_PATH` below. The first run downloads the pretrained checkpoint.

In [ ]:
%pip install "torch>=2.6,<3" "transformers==4.57.6" "accelerate>=1.10,<2" "sentencepiece>=0.2,<0.3" "pandas>=2,<3" "scikit-learn>=1.4,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 129.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.8 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency co

## Configuration

Set `TEST_FRACTION` to change the share of unique texts held out for testing.
The default is 80% training and 20% testing. If you change the split, rerun
from this cell through training so the weights correspond to that split.
Use the same software versions, hardware, and settings when repeating a run.

In [ ]:
from pathlib import Path

DATA_PATH = Path("/content/aggr_ds.csv")
TEST_FRACTION = 0.20
SAVE_DIR = Path("deberta-v3-base-emotion")

SEED = 42
MODEL_ID = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
EPOCHS = 5
LEARNING_RATE = 2e-5 # a better rate for _fine tuning_
MAX_STEPS = -1  # Keep -1 for full training, or set a small positive value to try the pipeline.

EMOTIONS = ("Anger", "Anticipation", "Disgust", "Fear", "Joy", "Sadness", "Surprise", "Trust")
LABELS = (*EMOTIONS, "Neutral")  # the model has one output per label: 9 in total
label_to_id = {label: index for index, label in enumerate(LABELS)}
id_to_label = dict(enumerate(LABELS))

# generated with pandas and pprint
MAPPINGS = {
    'sadness': ['sadness', None],
    'nan': ['neutral', None],
    'neutral': ['neutral', None],
    'love': ['joy', 'trust'],
    'gratitude': ['joy', 'trust'],
    'disapproval': ['anger', 'disgust'],
    'amusement': ['joy', None],
    'disappointment': ['sadness', 'surprise'],
    'disgust': ['disgust', None],
    'admiration': ['trust', None],
    'realization': ['surprise', None],
    'annoyance': ['anger', None],
    'confusion': ['surprise', 'surprise'],
    'optimism': ['anticipation', 'joy'],
    'curiosity': ['anticipation', 'trust'],
    'excitement': ['joy', 'anticipation'],
    'caring': ['trust', 'trust'],
    'joy': ['joy', None],
    'remorse': ['sadness', 'disgust'],
    'approval': ['trust', 'trust'],
    'nervousness': ['fear', 'fear'],
    'embarrassment': ['fear', 'disgust'],
    'surprise': ['surprise', None],
    'anger': ['anger', None],
    'grief': ['sadness', None],
    'pride': ['joy', 'trust'],
    'desire': ['anticipation', 'anticipation'],
    'relief': ['joy', 'joy'],
    'fear': ['fear', None],
    'trust': ['trust', None],
    'anticipation': ['anticipation', None],
    'pessimism': ['sadness', 'anticipation']
 }

In [ ]:
from google.colab import files

uploaded = files.upload()
# assert len(uploaded) == 1

!unzip "aggr_ds.csv.zip"

Saving aggr_ds.csv.zip to aggr_ds.csv.zip
Archive:  aggr_ds.csv.zip
  inflating: aggr_ds.csv             
  inflating: __MACOSX/._aggr_ds.csv  


## Prepare the data and split by text

Each CSV row is one annotation of a text, with one or two labels in `emo_1`
and `emo_2`; `Neutral` is kept as the ninth label. Rows are grouped by text,
so each distinct text becomes exactly one example. Its target for each label
is the fraction of that text's rows that give the label: a text annotated
three times with Joy, Joy and Joy + Trust gets Joy 1.0 and Trust 0.33. A text
with a single row gets 1.0 for each label in that row.

Case, Unicode, and whitespace variants count as the same text, so all
annotations of a text stay in the same split. The model receives the first
variant seen, with leading and trailing whitespace removed.

In [ ]:
import math
import random
import unicodedata
from collections import Counter, defaultdict

import pandas as pd

text_column = "text"
label_columns = ["emo_1", "emo_2"]  # emo_2 may be empty; src and base are ignored
canonical = {label.casefold(): label for label in LABELS}  # "joy" and "Joy" both work

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    usecols=[text_column, *label_columns],
    dtype=str,
    keep_default_na=False,  # keep empty cells as "" instead of NaN
)
df = df[[text_column, *label_columns]].apply(lambda col: col.str.strip())

empty_text = df.index[df[text_column] == ""]
if len(empty_text):
    raise ValueError(f"Empty text in data rows {(empty_text + 1).tolist()[:10]} (1-based, header excluded).")
empty_emo_1 = df.index[df["emo_1"] == ""]
if len(empty_emo_1):
    raise ValueError(f"Empty emo_1 in data rows {(empty_emo_1 + 1).tolist()[:10]} (1-based, header excluded).")
unknown = sorted({value for value in df[label_columns].to_numpy().ravel() if value and value.casefold() not in canonical})
if unknown:
    raise ValueError(f"Unknown labels {unknown}. Expected {list(LABELS)}.")

# Group rows by text and count how many rows give each label.
first_text = {}
rows_per_text = Counter()
label_votes = defaultdict(Counter)
for text, *raw_labels in df.itertuples(index=False):
    text_key = " ".join(unicodedata.normalize("NFKC", text).casefold().split())
    first_text.setdefault(text_key, text)
    rows_per_text[text_key] += 1
    for label in {canonical[raw.casefold()] for raw in raw_labels if raw}:
        label_votes[text_key][label] += 1

# One example per text: (text, 9 target values between 0 and 1).
examples = [
    (text, [label_votes[key][label] / rows_per_text[key] for label in LABELS])
    for key, text in first_text.items()
]

repeated = sum(count > 1 for count in rows_per_text.values())
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print(f"Grouped them into {len(examples):,} distinct texts; {repeated:,} texts appear in more than one row.")

Loaded 269,192 rows from /content/aggr_ds.csv
Grouped them into 64,457 distinct texts; 63,226 texts appear in more than one row.


In [ ]:
if not 0 < TEST_FRACTION < 1:
    raise ValueError("TEST_FRACTION must be between 0 and 1.")

shuffled = examples[:]
random.Random(SEED).shuffle(shuffled)
n_test = math.ceil(TEST_FRACTION * len(shuffled))
if not 0 < n_test < len(shuffled):
    raise ValueError("Adjust TEST_FRACTION so each split contains at least one text.")
test_examples, train_examples = shuffled[:n_test], shuffled[n_test:]


def texts_per_label(split):
    """Number of texts in the split with a nonzero target for each label."""
    return [sum(target[index] > 0 for _, target in split) for index in id_to_label]


train_counts, test_counts = texts_per_label(train_examples), texts_per_label(test_examples)
for split_name, counts in (("train", train_counts), ("test", test_counts)):
    missing = [id_to_label[index] for index, count in enumerate(counts) if not count]
    if missing:
        raise ValueError(f"The {split_name} split is missing {missing}. Change SEED or TEST_FRACTION.")

print(f"Train: {len(train_examples):,} texts | Test: {len(test_examples):,} texts")
print(f"{'Label':<15}{'Train':>9}{'Test':>9}   (texts with a nonzero target)")
for index, label in id_to_label.items():
    print(f"{label:<15}{train_counts[index]:>9,}{test_counts[index]:>9,}")

Train: 51,565 texts | Test: 12,892 texts
Label              Train     Test   (texts with a nonzero target)
Anger             16,179    4,130
Anticipation      15,888    3,948
Disgust           12,913    3,235
Fear               4,772    1,238
Joy               21,111    5,227
Sadness           10,028    2,642
Surprise          15,565    3,920
Trust             26,185    6,561
Neutral           26,093    6,496


## Fine-tune and save the model

The classifier has nine logits, one per label, and is trained with binary
cross-entropy against each text's nine target values. The training cell
saves the model weights, label mapping, and tokenizer to `SAVE_DIR`. Choose
a different `SAVE_DIR` to keep multiple runs.

In [ ]:
import pandas as pd
import torch
import transformers
from IPython.display import display
from sklearn.metrics import classification_report
from torch.nn import functional as F
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer):
        self.encodings = tokenizer([text for text, _ in examples], truncation=True)
        self.labels = [target for _, target in examples]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = self.labels[index]  # 9 floats -> float tensor in the collator
        return item

In [ ]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=False,
    model_max_length=MAX_LENGTH,
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    num_labels=len(LABELS),                       # 9
    id2label=id_to_label,                         # must also have 9 entries: it overrides num_labels
    label2id=label_to_id,
    problem_type="multi_label_classification",    # sigmoid + BCEWithLogitsLoss
    use_safetensors=False,  # The pinned Microsoft checkpoint provides PyTorch .bin weights.
)

if model.config.num_labels != len(LABELS):
    raise ValueError(f"Model has {model.config.num_labels} outputs; expected {len(LABELS)}.")

train_dataset = EmotionDataset(train_examples, tokenizer)
test_dataset = EmotionDataset(test_examples, tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Warm up over the first 6% of steps. Transformers v5 removed warmup_ratio and takes the
# fraction as warmup_steps; v4 treats warmup_steps as a whole number of steps.
if int(transformers.__version__.split(".")[0]) >= 5:
    warmup = {"warmup_steps": 0.06}
else:
    warmup = {"warmup_ratio": 0.06}

training_args = TrainingArguments(
    output_dir=str(SAVE_DIR),
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,          # if > 0, this overrides num_train_epochs
    learning_rate=LEARNING_RATE,  # 2e-5 to 3e-5 is a safe range for fine-tuning
    weight_decay=0.01,
    optim="adamw_torch",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    logging_nan_inf_filter=False,  # show NaN/inf losses instead of hiding them as 0
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False),
    report_to="none",
    **warmup,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=collator,
)
print(f"Training device: {trainer.args.device}")
trainer.train()
trainer.save_model(str(SAVE_DIR))
print(f"Saved model and tokenizer to {SAVE_DIR.resolve()}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

... (2 outputs left out)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Training device: cuda:0


Step,Training Loss
100,0.685600
200,0.490700
300,0.387900
400,0.372500
500,0.352000
600,0.342000
700,0.336200
800,0.330900
900,0.317400
1000,0.314600


Saved model and tokenizer to /content/deberta-v3-base-emotion


## Evaluate the held-out test set

A label counts as **predicted** when its confidence is at least `THRESHOLD`
(0.5 by default), and as **gold** when at least that fraction of the text's
annotations gave it. Precision, recall, and F1 are computed per label from
those yes/no decisions, with micro, macro, and weighted averages.

The summary also shows `bce_loss`, the binary cross-entropy against the
target values (the training loss; lower is better), `exact_match`, the share
of texts whose predicted label set equals the gold set exactly,
`top_label_correct`, the share of texts whose highest-confidence label was
given by at least one annotator, and `mean_abs_error`, the average gap
between a confidence and its target value.

In [ ]:
THRESHOLD = 0.5  # confidence at or above this counts as "label present"


def classification_metrics(logits, targets, label_order, threshold=THRESHOLD):
    scores = torch.as_tensor(logits, dtype=torch.float32, device="cpu")
    targets = torch.as_tensor(targets, dtype=torch.float32, device="cpu")
    confidences = scores.sigmoid()
    predicted = confidences >= threshold
    gold = targets >= threshold
    report = classification_report(
        gold.numpy().astype(int),
        predicted.numpy().astype(int),
        labels=list(range(len(label_order))),
        target_names=list(label_order),
        output_dict=True,
        zero_division=0,
    )
    report_rows = [*label_order, "micro avg", "macro avg", "weighted avg"]
    by_label = pd.DataFrame([report[row] for row in report_rows], index=report_rows)
    by_label["support"] = by_label["support"].astype(int)
    top_label = confidences.argmax(dim=-1, keepdim=True)
    summary = pd.Series({
        "bce_loss": F.binary_cross_entropy_with_logits(scores, targets).item(),
        "exact_match": (predicted == gold).all(dim=-1).float().mean().item(),
        "top_label_correct": (targets.gather(-1, top_label) > 0).float().mean().item(),
        "mean_abs_error": (confidences - targets).abs().mean().item(),
    })
    return summary, by_label


def labels_at_or_above(values, label_order, threshold=THRESHOLD):
    """Comma-separated labels whose value is at least the threshold, or a dash if none."""
    names = [label for label, value in zip(label_order, values.tolist()) if value >= threshold]
    return ", ".join(names) or "—"


def confidence_table(texts, confidences, label_order, threshold=THRESHOLD):
    table = pd.DataFrame(confidences.numpy(), columns=list(label_order))
    table.insert(0, "top_label", [label_order[index] for index in confidences.argmax(dim=-1).tolist()])
    table.insert(0, "predicted", [labels_at_or_above(row, label_order, threshold) for row in confidences])
    table.insert(0, "text", list(texts))
    return table


@torch.inference_mode()
def predict_logits(model, tokenizer, texts, batch_size=16):
    model.eval()
    device = next(model.parameters()).device
    batches = []
    for start in range(0, len(texts), batch_size):
        inputs = tokenizer(
            list(texts[start:start + batch_size]),
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)
        batches.append(model(**inputs).logits.float().cpu())
    return torch.cat(batches)

In [ ]:
test_output = trainer.predict(test_dataset, metric_key_prefix="test")
test_logits, test_targets = test_output.predictions, test_output.label_ids

test_summary, test_report = classification_metrics(test_logits, test_targets, LABELS)
display(test_summary.round(4))
display(test_report.round(4))

,0
bce_loss,0.2888
exact_match,0.4645
top_label_correct,0.8670
mean_abs_error,0.1013


,precision,recall,f1-score,support
Anger,0.5936,0.4747,0.5275,1403
Anticipation,0.6014,0.3886,0.4721,1145
Disgust,0.5118,0.2687,0.3524,804
Fear,0.6160,0.5329,0.5714,289
Joy,0.7868,0.7808,0.7838,2760
Sadness,0.6667,0.5777,0.6190,779
Surprise,0.5362,0.2913,0.3775,738
Trust,0.7424,0.6844,0.7122,3099
Neutral,0.6808,0.3629,0.4734,2921
micro avg,0.6952,0.5368,0.6058,13938


## Evaluate selected test examples

Leave `SELECTED_TEST_INDICES` as `None` to sample, with seed 42, one held-out
text for each label that has that label in its gold set. The table index
gives the zero-based position in the held-out split. Enter your own list of
positions to score that subset. Per-label metrics with undefined
denominators are shown as zero.

In [ ]:
SELECTED_TEST_INDICES = None  # Example: [0, 12, 34]

target_tensor = torch.as_tensor(test_targets, dtype=torch.float32)
if SELECTED_TEST_INDICES is None:
    example_rng = random.Random(SEED)
    selected = []
    for index in id_to_label:
        candidates = torch.nonzero(target_tensor[:, index] >= THRESHOLD).flatten().tolist()
        if candidates:
            selected.append(example_rng.choice(candidates))
    selected = list(dict.fromkeys(selected))  # drop repeats if one text was picked for two labels
else:
    selected = list(SELECTED_TEST_INDICES)
if not selected or any(index < 0 or index >= len(test_examples) for index in selected):
    raise ValueError(f"Choose one or more test positions from 0 to {len(test_examples) - 1}.")

selected_logits, selected_targets = test_logits[selected], test_targets[selected]
selected_confidences = torch.as_tensor(selected_logits).float().sigmoid()
selected_table = confidence_table(
    [test_examples[index][0] for index in selected],
    selected_confidences,
    LABELS,
)
selected_table.insert(1, "gold", [labels_at_or_above(target_tensor[index], LABELS) for index in selected])
selected_table.index = pd.Index(selected, name="test_index")
display(selected_table.round(4))

selected_summary, selected_report = classification_metrics(selected_logits, selected_targets, LABELS)
display(selected_summary.round(4))
display(selected_report.round(4))

## Reload the saved model and try your own examples

After training, run the cells below to load the saved files and score new
words or sentences. In a fresh session, run setup, configuration, the
imports cell, and the cell that defines `classification_metrics` and
`predict_logits`. Then run the cells below with `SAVE_DIR` set to the
saved model folder.

In [ ]:
saved_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=False, local_files_only=True)
saved_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR, use_safetensors=True, local_files_only=True
).eval()
saved_labels = tuple(
    saved_model.config.id2label[index] for index in range(saved_model.config.num_labels)
)

In [ ]:
CUSTOM_TEXTS = ["delighted", "I jumped when the lights suddenly went out."]
CUSTOM_GOLD = None  # Set to one list of labels per text, such as [["Joy"], ["Surprise", "Fear"]], to score these.

if not CUSTOM_TEXTS or any(not text.strip() for text in CUSTOM_TEXTS):
    raise ValueError("Enter at least one word or sentence, with no empty items.")

custom_logits = predict_logits(saved_model, saved_tokenizer, CUSTOM_TEXTS)
custom_confidences = custom_logits.sigmoid()  # Shape: (number of texts, 9), each value between 0 and 1.
display(confidence_table(CUSTOM_TEXTS, custom_confidences, saved_labels).round(4))

if CUSTOM_GOLD is not None:
    label2id = saved_model.config.label2id
    if len(CUSTOM_GOLD) != len(CUSTOM_TEXTS) or any(
        isinstance(labels, str) or any(label not in label2id for label in labels) for labels in CUSTOM_GOLD
    ):
        raise ValueError(f"Provide one list of labels per text, using labels from {saved_labels}.")
    custom_targets = torch.zeros(len(CUSTOM_TEXTS), len(saved_labels))
    for row, labels in enumerate(CUSTOM_GOLD):
        for label in labels:
            custom_targets[row, label2id[label]] = 1.0
    custom_summary, custom_report = classification_metrics(custom_logits, custom_targets, saved_labels)
    display(custom_summary.round(4))
    display(custom_report.round(4))

[Microsoft DeBERTa-v3-base model card](https://huggingface.co/microsoft/deberta-v3-base) ·
[Hugging Face Trainer documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)